## Audio books analysis
This model will analyse the data of past clients to train and then be able to forecast whether a customer is likely to buy again or not.

## My take at the preprocessing

In [1]:
import numpy as np
from sklearn import preprocessing

### Load the data

In [2]:
raw_csv_data = np.loadtxt('../../../../statistics/python/audiobooks/Audiobooks_data.csv', delimiter=',')
unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:,-1]

### Balance the dataset
There are far too many '0' compared to '1'. We will use all '1' and take as many '0' as we did '1' to have the same amount of data to train the model on recongnising both '0' and '1' equaly.

In [3]:
#Obtain the different classes from the targets column
target_classes,counts=np.unique(targets_all, return_counts=True)
# number of samples in the smallest data class, to limit all other classes to this number of samples
num_samples_one_class = counts.min()

samples_indices_dict = {}  #will contain the "counts.min()" indices for each of the target classes
for i in target_classes:
            # get indices for all rows of one class (target column with one value)
    class_indices = np.where(targets_all == i)[0]
            # shuffle all indices of this class to take a random sample of the class
    np.random.shuffle(class_indices)
            # take a sample of num_samples_one_class row indices
    samples_indices_dict[i] = class_indices[:num_samples_one_class]
    print(samples_indices_dict[i])
    print(len(samples_indices_dict[i]))

     #join indices of all classes to have the "complete data" to use in the process
balanced_indices = np.concatenate(list(samples_indices_dict.values()))

[ 4043 10476  1651 ...  6929  7366 13998]
2237
[12020 10807  2382 ...  1600   468 13187]
2237


### Shuffle the data

In [4]:
#shuffle the data by shuffling the indices and then "slice" data and targets with the indices
shuffled_indices = np.copy(balanced_indices)
np.random.shuffle(shuffled_indices)
shuffled_targets = targets_all[shuffled_indices]
shuffled_inputs = unscaled_inputs_all[shuffled_indices]

### Standardize the inputs

In [5]:
#Standardize the inputs
scaled_inputs = preprocessing.scale(shuffled_inputs)

### Split in train, validate and test

In [6]:
num_samples = scaled_inputs.shape[0]
num_train_samples = int(num_samples * 0.8)
num_validate_samples = int (num_samples * 0.1)
num_test_samples = num_samples - num_train_samples - num_validate_samples

train_inputs = scaled_inputs[:num_train_samples,:]
train_targets = shuffled_targets[:num_train_samples]

validate_inputs = scaled_inputs[num_train_samples:num_train_samples + num_validate_samples,:]
validate_targets = shuffled_targets[num_train_samples:num_train_samples + num_validate_samples]

test_inputs = scaled_inputs[num_train_samples + num_validate_samples:,:]
test_targets = shuffled_targets[num_train_samples + num_validate_samples:]

# Check that the '0'- and '1'-classes are balanced in each of the three datasets
print(np.sum(train_targets), num_train_samples, np.sum(train_targets)/num_train_samples)
print(np.sum(validate_targets), num_validate_samples, np.sum(validate_targets)/num_validate_samples)
print(np.sum(test_targets), num_test_samples, np.sum(test_targets)/num_test_samples)


1788.0 3579 0.49958088851634536
232.0 447 0.5190156599552572
217.0 448 0.484375


### Save data separately in *.npz

In [7]:
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_validation', inputs=validate_inputs, targets=validate_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

### Machine learing part
Preprocesing is done. It can be reused for other problems.
From here on, we will start from the .npz files. It will be done in a different notebook.